In [18]:
# =============================================================================
# Cell 1 — imports + helper functions.  Run this once at the start of a session.
# =============================================================================
%gui qt  # let napari's Qt event loop run alongside the Jupyter kernel

from pathlib import Path

import cc3d
import napari
import numpy as np
import tifffile
from scipy.ndimage import binary_dilation


def _resolve_start(shape, size, place):
    """Return the (z0, y0, x0) start of a `size` cube in a volume of `shape`."""
    if isinstance(place, str):
        if place == 'center':
            return tuple((s - size) // 2 for s in shape)
        if place == 'origin':
            return (0, 0, 0)
        if place == 'farcorner':
            return tuple(s - size for s in shape)
        raise ValueError(f'Unknown place keyword: {place!r}')
    return tuple(int(v) for v in place)


def load_full(path):
    """Load an entire 3D volume from .tif/.tiff or .npy."""
    p = Path(path)
    ext = p.suffix.lower()
    if ext in ('.tif', '.tiff'):
        return tifffile.imread(str(p))
    if ext == '.npy':
        return np.load(p)
    raise ValueError(f'Unsupported file type: {ext}')


def load_crop(path, size, place):
    """Read only a (size, size, size) cube from disk.

    .tif : reads just the Z-pages we need (page-by-page).
    .npy : uses np.load(mmap_mode='r') and copies out the requested slice.
    """
    p = Path(path)
    ext = p.suffix.lower()
    if ext in ('.tif', '.tiff'):
        with tifffile.TiffFile(str(p)) as tif:
            series = tif.series[0]
            shape = series.shape
            dtype = series.dtype
            start = _resolve_start(shape, size, place)
            z0, y0, x0 = start
            out = np.empty((size, size, size), dtype=dtype)
            for i, z in enumerate(range(z0, z0 + size)):
                page = tif.pages[z].asarray()
                out[i] = page[y0 : y0 + size, x0 : x0 + size]
            return out, start
    if ext == '.npy':
        mm = np.load(p, mmap_mode='r')
        start = _resolve_start(mm.shape, size, place)
        z0, y0, x0 = start
        return np.asarray(
            mm[z0 : z0 + size, y0 : y0 + size, x0 : x0 + size]
        ), start
    raise ValueError(f'Unsupported file type: {ext}')


def peek_shape(path):
    """Return the shape of a 3D file on disk without loading its contents."""
    p = Path(path)
    ext = p.suffix.lower()
    if ext in ('.tif', '.tiff'):
        with tifffile.TiffFile(str(p)) as tif:
            return tif.series[0].shape
    if ext == '.npy':
        return np.load(p, mmap_mode='r').shape
    raise ValueError(f'Unsupported file type: {ext}')


def relabel_sequential_simple(labels):
    """Remap labels to [0, 1, ..., n] preserving order of first appearance."""
    unique = np.unique(labels)
    remap = np.zeros(int(unique.max()) + 1, dtype=labels.dtype)
    remap[unique] = np.arange(len(unique), dtype=labels.dtype)
    return remap[labels]


print('[cell 1] Functions defined. Ready.')

ERROR:root:Invalid GUI request "qt # let napari's Qt event loop run alongside the Jupyter kernel", valid ones are:dict_keys(['inline', 'nbagg', 'webagg', 'notebook', 'ipympl', 'widget', None, 'qt', 'qt5', 'qt6', 'wx', 'tk', 'gtk', 'gtk3', 'osx', 'asyncio'])


[cell 1] Functions defined. Ready.


In [19]:
# =============================================================================
# Cell 2 — load and crop.  Slow only the first time (disk I/O); re-run after
# changing IMAGE_PATH / MASK_PATH / CUBE_SIZE / CROP_POSITION.
# =============================================================================

# ---- Input ------------------------------------------------------------------
IMAGE_PATH = Path(r'D:/Chenhao_Napari_Plugin/Data/image_volume.tif')
MASK_PATH = Path(r'D:/Chenhao_Napari_Plugin/Data/intracristal_volume_v3.npy')

# ---- Cropping ---------------------------------------------------------------
# CUBE_SIZE = None -> keep the full volume.
CUBE_SIZE = 500
# "center" | "origin" | "farcorner"  |  or an explicit (z0, y0, x0) tuple.
CROP_POSITION = 'center'

# -----------------------------------------------------------------------------
print('[cell 2 / 1] Inspecting file shapes...')
image_shape = peek_shape(IMAGE_PATH)
mask_shape = peek_shape(MASK_PATH)
print(f'           image on disk: shape={image_shape}')
print(f'           mask  on disk: shape={mask_shape}')
assert image_shape == mask_shape, 'Image and mask must have identical shape.'

if CUBE_SIZE is None or all(s <= CUBE_SIZE for s in image_shape):
    print('[cell 2 / 2] Loading full image (no crop needed)...')
    image = load_full(IMAGE_PATH)
    print('[cell 2 / 3] Loading full mask ...')
    mask = load_full(MASK_PATH)
    start = (0, 0, 0)
else:
    print(
        f"[cell 2 / 2] Cropping image ({CUBE_SIZE}^3 at '{CROP_POSITION}') ..."
    )
    image, start = load_crop(IMAGE_PATH, CUBE_SIZE, CROP_POSITION)
    print(
        f"[cell 2 / 3] Cropping mask  ({CUBE_SIZE}^3 at '{CROP_POSITION}') ..."
    )
    mask, _ = load_crop(MASK_PATH, CUBE_SIZE, CROP_POSITION)

mask_bin = (mask > 0).astype(np.uint8)

print(
    f'[cell 2 / done] image={image.shape} {image.dtype}, '
    f'mask_bin coverage={mask_bin.mean() * 100:.2f}%  (start={start})'
)

[cell 2 / 1] Inspecting file shapes...
           image on disk: shape=(1065, 1536, 2048)
           mask  on disk: shape=(1065, 1536, 2048)
[cell 2 / 2] Cropping image (500^3 at 'center') ...
[cell 2 / 3] Cropping mask  (500^3 at 'center') ...
[cell 2 / done] image=(500, 500, 500) uint8, mask_bin coverage=0.38%  (start=(282, 518, 774))


In [20]:
# =============================================================================
# Cell 3 — dilate + connected components + sequential relabel + napari.
#
# The heavy step (binary_dilation + cc3d) is cached against
# (mask_bin identity, DILATION_ITERATIONS, CONNECTIVITY); re-running this cell
# with the same parameters skips it.  Cells 4 and 5 filter the result.
# =============================================================================

# ---- Labelling --------------------------------------------------------------
DILATION_ITERATIONS = 7  # more -> nearby cristae get merged into one label
CONNECTIVITY = 26  # 6, 18, or 26 for 3D

# -----------------------------------------------------------------------------
_label_key = (id(mask_bin), DILATION_ITERATIONS, CONNECTIVITY)
if globals().get('_last_label_key') != _label_key:
    print(f'[cell 3 / 1] Dilating (iterations={DILATION_ITERATIONS}) ...')
    dilated = binary_dilation(mask_bin, iterations=DILATION_ITERATIONS)
    print(
        f'[cell 3 / 2] Connected components (connectivity={CONNECTIVITY}) ...'
    )
    cc = cc3d.connected_components(
        dilated.astype(np.uint8), connectivity=CONNECTIVITY
    )
    print(
        '[cell 3 / 3] Restoring binary mask geometry + sequential relabel ...'
    )
    labels_all = relabel_sequential_simple((cc * mask_bin).astype(np.int32))
    _last_label_key = _label_key
    print(f'             -> {int(labels_all.max())} label IDs')
else:
    print(
        '[cell 3 / 1-3] Cached labels_all (dilation + CC parameters unchanged).'
    )

# Downstream defaults — cells 4/5 may overwrite these.
image_out = image
labels_out = labels_all

print('[cell 3 / 4] Updating napari viewer ...')
viewer = napari.current_viewer() or napari.Viewer()
viewer.layers.clear()
viewer.add_image(image, name='image')
viewer.add_labels(labels_all.astype(np.int32), name='labels_all')

unique_nonzero = np.unique(labels_all)
unique_nonzero = unique_nonzero[unique_nonzero != 0]
print(f'[cell 3 / done] Available label IDs ({len(unique_nonzero)}):')
print(unique_nonzero.tolist())

[cell 3 / 1] Dilating (iterations=7) ...
[cell 3 / 2] Connected components (connectivity=26) ...
[cell 3 / 3] Restoring binary mask geometry + sequential relabel ...
             -> 31 label IDs
[cell 3 / 4] Updating napari viewer ...
[cell 3 / done] Available label IDs (31):
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]


In [21]:
# =============================================================================
# Cell 4 — delete unwanted labels, sequentially renumber the survivors,
#          and drop the result into napari as `labels_filtered`.
#
# Iterate freely here: this cell is fast (no dilation, no CC).
# =============================================================================

# ---- Deletion ---------------------------------------------------------------
# List the label IDs you want to REMOVE (see the printout in cell 3).
# Leave empty ([]) to keep everything.
DELETE_LABELS = [
    1,
    8,
    12,
    13,
    16,
    17,
    18,
    22,
    24,
    29,
    30,
    31,
]  # e.g. [1, 3, 7]

# -----------------------------------------------------------------------------
print('[cell 4 / 1] Removing labels ...')
if not DELETE_LABELS:
    labels_out = labels_all.copy()
    print('             DELETE_LABELS is empty -> nothing removed.')
else:
    delete_arr = np.asarray(
        sorted({int(v) for v in DELETE_LABELS}), dtype=labels_all.dtype
    )
    unique_all = np.unique(labels_all)
    missing = [int(k) for k in delete_arr if k not in unique_all]
    if missing:
        print(
            f'             Warning: labels {missing} not in the volume - ignored.'
        )
    del_mask = np.isin(labels_all, delete_arr)
    labels_out = np.where(del_mask, 0, labels_all).astype(labels_all.dtype)
    print(f'             Removed {delete_arr.tolist()}.')

print('[cell 4 / 2] Sequential relabel to [0, 1, ..., n] ...')
labels_out = relabel_sequential_simple(labels_out)
print(f'             -> {int(labels_out.max())} label IDs remain.')

# labels_out was just refreshed; keep image_out as the original crop.
image_out = image

print('[cell 4 / 3] Updating napari viewer ...')
viewer = napari.current_viewer() or napari.Viewer()
if 'labels_filtered' in viewer.layers:
    del viewer.layers['labels_filtered']
viewer.add_labels(labels_out.astype(np.int32), name='labels_filtered')
print('[cell 4 / done]')

[cell 4 / 1] Removing labels ...
             Removed [1, 8, 12, 13, 16, 17, 18, 22, 24, 29, 30, 31].
[cell 4 / 2] Sequential relabel to [0, 1, ..., n] ...
             -> 19 label IDs remain.
[cell 4 / 3] Updating napari viewer ...
[cell 4 / done]


In [22]:
# =============================================================================
# Cell 5 — keep only selected labels, tight-crop image + mask around them,
#          resequentially renumber, and add to napari.
#
# Acts on the CURRENT `labels_out` (i.e. after cell 4 if you ran it, otherwise
# the full labelling from cell 3).  The output pair (`image_out`, `labels_out`)
# is what cell 6 saves.
# =============================================================================

# ---- Selection --------------------------------------------------------------
# Label IDs to KEEP (see cell 3's printout, or the napari layer after cell 4).
# Leave empty ([]) to keep every currently-active label.
KEEP_LABELS = [11, 13, 14, 17, 19]  # e.g. [2, 5]

# Voxels of empty space to leave around the tight bounding box.  For the
# persistent-homology dilation mode, ensure PAD >= max_steps * Lambda voxels
# so the dilation can expand into empty space rather than hitting the volume
# edge (with defaults Lambda=0.1, max_steps=100 that means PAD >= 10).
PAD = 20

# -----------------------------------------------------------------------------
print('[cell 5 / 1] Filtering to KEEP_LABELS ...')
if not KEEP_LABELS:
    labels_kept = labels_out.copy()
    print(
        '             KEEP_LABELS is empty -> keeping every currently-active label.'
    )
else:
    keep_arr = np.asarray(
        sorted({int(v) for v in KEEP_LABELS}), dtype=labels_out.dtype
    )
    unique = np.unique(labels_out)
    missing = [int(k) for k in keep_arr if k not in unique]
    if missing:
        print(
            f'             Warning: labels {missing} not in current labels_out - ignored.'
        )
    keep_mask = np.isin(labels_out, keep_arr)
    labels_kept = np.where(keep_mask, labels_out, 0).astype(labels_out.dtype)
    print(f'             Kept {keep_arr.tolist()}.')

if int(labels_kept.max()) == 0:
    raise ValueError(
        'No voxels remain after KEEP_LABELS filter — nothing to crop.'
    )

print(f'[cell 5 / 2] Computing tight bounding box (+ pad={PAD} voxels) ...')
nonzero = np.argwhere(labels_kept > 0)
zyx_min = nonzero.min(axis=0)
zyx_max = nonzero.max(axis=0) + 1  # exclusive
shape = labels_kept.shape
z0 = max(int(zyx_min[0]) - PAD, 0)
y0 = max(int(zyx_min[1]) - PAD, 0)
x0 = max(int(zyx_min[2]) - PAD, 0)
z1 = min(int(zyx_max[0]) + PAD, shape[0])
y1 = min(int(zyx_max[1]) + PAD, shape[1])
x1 = min(int(zyx_max[2]) + PAD, shape[2])
print(f'             bbox=Z[{z0}:{z1}] Y[{y0}:{y1}] X[{x0}:{x1}]')

print('[cell 5 / 3] Cropping image + labels and sequential relabel ...')
image_cropped = image[z0:z1, y0:y1, x0:x1].copy()
labels_cropped = relabel_sequential_simple(labels_kept[z0:z1, y0:y1, x0:x1])
print(
    f'             new shape={image_cropped.shape}, '
    f'{int(labels_cropped.max())} labels remain after renumber'
)

# Cell 6 saves these two.
image_out = image_cropped
labels_out = labels_cropped

print(
    '[cell 5 / 4] Updating napari viewer '
    '(positioned at original coords via `translate`) ...'
)
viewer = napari.current_viewer() or napari.Viewer()
for name in ('image_cropped', 'labels_cropped'):
    if name in viewer.layers:
        del viewer.layers[name]
viewer.add_image(image_cropped, name='image_cropped', translate=(z0, y0, x0))
viewer.add_labels(
    labels_cropped.astype(np.int32),
    name='labels_cropped',
    translate=(z0, y0, x0),
)
print('[cell 5 / done]')

[cell 5 / 1] Filtering to KEEP_LABELS ...
             Kept [11, 13, 14, 17, 19].
[cell 5 / 2] Computing tight bounding box (+ pad=20 voxels) ...
             bbox=Z[189:500] Y[25:208] X[305:500]
[cell 5 / 3] Cropping image + labels and sequential relabel ...
             new shape=(311, 183, 195), 5 labels remain after renumber
[cell 5 / 4] Updating napari viewer (positioned at original coords via `translate`) ...
[cell 5 / done]


In [23]:
# =============================================================================
# Cell 6 — save (image_out + labels_out).
#
# What gets written depends on which cells you ran last:
#   - Only cell 3            -> full crop + all labels
#   - Cells 3, 4             -> full crop + (all labels minus DELETE_LABELS)
#   - Cells 3 [, 4], 5       -> tight-cropped image + selected labels
#
# Label dtype is downcast to uint8 / uint16 / uint32.
# =============================================================================

OUTPUT_DIR = Path(r'D:/Chenhao_Napari_Plugin/Data')
OUTPUT_NAME = 'cristae_prepped_5labels'

# -----------------------------------------------------------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
image_path = OUTPUT_DIR / f'{OUTPUT_NAME}_image.tif'
labels_path = OUTPUT_DIR / f'{OUTPUT_NAME}_labels.npy'

max_id = int(labels_out.max())
if max_id < 2**8:
    save_dtype = np.uint8
elif max_id < 2**16:
    save_dtype = np.uint16
else:
    save_dtype = np.uint32

print(
    f'[cell 6 / 1] Writing image  -> {image_path}  (shape={image_out.shape})'
)
tifffile.imwrite(str(image_path), image_out)
print(
    f'[cell 6 / 2] Writing labels -> {labels_path}  '
    f'(dtype={save_dtype.__name__}, max_id={max_id})'
)
np.save(labels_path, labels_out.astype(save_dtype))
print('[cell 6 / done]')

[cell 6 / 1] Writing image  -> D:\Chenhao_Napari_Plugin\Data\cristae_prepped_5labels_image.tif  (shape=(311, 183, 195))
[cell 6 / 2] Writing labels -> D:\Chenhao_Napari_Plugin\Data\cristae_prepped_5labels_labels.npy  (dtype=uint8, max_id=5)
[cell 6 / done]


c:\Users\Administrator\anaconda3\envs\napari-persistent-homology\Lib\site-packages\napari\layers\_layer_actions.py:85: UserWarning: projection mode "mean" is not compatible with Labels layers. Falling back to "none".
  warnings.warn(
